# Módulo 4 — Gemini API: Análise Estruturada de Dados

Neste notebook usamos a **API do Google Gemini** via SDK Python para análises que retornam dados estruturados (JSON), ideal para integrar IA em pipelines de dados.

## O que vamos construir

1. Conexão com a API usando `google-genai` SDK
2. Análise de um registro de consumidor com retorno forçado em JSON
3. Processamento em lote: analisar múltiplos registros
4. Geração de relatório narrativo executivo
5. Conversa multi-turno com histórico de mensagens

> **Pré-requisito:** `GEMINI_API_KEY` configurada no arquivo `.env` na raiz do projeto.

In [ ]:
import os
import json
import sys
import time
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../../.env"))

try:
    from google import genai
    from google.genai import types
    print("google-genai: OK")
except ImportError:
    print("Instale com: uv add google-genai")
    sys.exit(1)

import pandas as pd

# Base de dados
possible = [
    "../../modulo2_dados_com_pandas/dados/consumidores_simulado.csv",
    "../modulo2_dados_com_pandas/dados/consumidores_simulado.csv",
]
csv_path = next((p for p in possible if Path(p).exists()), None)
df = pd.read_csv(csv_path) if csv_path else pd.DataFrame()

client = genai.Client()
MODEL = "gemini-2.0-flash"

print(f"Modelo: {MODEL}")
print(f"Base: {len(df)} registros" if not df.empty else "Base não encontrada")

## 1. Chamada básica — diagnóstico de um registro

Usamos `response_mime_type="application/json"` para forçar o Gemini a retornar JSON válido, sem texto adicional.

In [ ]:
# Pegar um registro com problemas da base
if not df.empty:
    registro = df.iloc[6].to_dict()  # registro com outlier de consumo
else:
    registro = {
        "cod_consumidor": 7,
        "nom_consumidor": "Consumidor Teste",
        "cpf_cnpj": None,
        "cod_modalidade_tarifaria": "B1",
        "cod_classe_consumo": "RESIDENCIAL",
        "vlr_consumo_medio_kwh": 9999999,
        "flg_ativo": True,
    }

prompt_diagnostico = f"""
Você é um especialista em qualidade de dados do setor elétrico brasileiro.

Analise o seguinte registro de uma unidade consumidora e identifique problemas:

{json.dumps(registro, ensure_ascii=False, indent=2, default=str)}

Retorne um JSON com exatamente esta estrutura:
{{
  "cod_consumidor": <int>,
  "score_qualidade": <0-100>,
  "problemas": [
    {{"campo": "<nome>", "severidade": "CRITICO|ALTO|MEDIO", "descricao": "<texto>"}}
  ],
  "recomendacao": "<ação corretiva principal>"
}}
"""

response = client.models.generate_content(
    model=MODEL,
    contents=prompt_diagnostico,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",  # força retorno JSON
        temperature=0.1,
    ),
)

print("Resposta bruta do Gemini:")
print(response.text)
print()

resultado = json.loads(response.text)
print("JSON parseado:")
print(json.dumps(resultado, ensure_ascii=False, indent=2))

## 2. Processamento em lote

Analisamos múltiplos registros em loop e consolidamos os resultados em um DataFrame.

In [ ]:
def analisar_registro(registro: dict) -> dict:
    """Envia um registro ao Gemini e retorna análise estruturada em JSON."""

    prompt = f"""Analise este registro de consumidor de energia elétrica.

{json.dumps(registro, ensure_ascii=False, default=str)}

Retorne JSON com esta estrutura exata:
{{
  "cod_consumidor": <int>,
  "score": <0-100>,
  "status": "OK|ATENCAO|CRITICO",
  "problemas": ["<problema1>", "<problema2>"],
  "acao_sugerida": "<texto curto>"
}}"""

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.1,
            ),
        )
        return json.loads(response.text)
    except Exception as e:
        return {
            "cod_consumidor": registro.get("cod_consumidor"),
            "score": 0,
            "status": "ERRO",
            "problemas": [str(e)],
            "acao_sugerida": "Verificar manualmente",
        }


# Processar os primeiros 5 registros
amostra = df.head(5).to_dict(orient="records") if not df.empty else [registro]

resultados = []
for rec in amostra:
    print(f"Analisando UC {rec.get('cod_consumidor')}...", end=" ")
    r = analisar_registro(rec)
    resultados.append(r)
    print(f"Score={r['score']} | {r['status']}")
    time.sleep(0.5)  # rate limit

df_resultados = pd.DataFrame(resultados)
print("\nResultados consolidados:")
print(df_resultados[["cod_consumidor", "score", "status", "acao_sugerida"]])

## 3. Relatório narrativo executivo

Consolidamos os dados e pedimos ao Gemini um relatório em linguagem natural, pronto para enviar à liderança.

In [ ]:
resumo_base = {
    "total_registros": len(df_resultados),
    "score_medio": round(df_resultados["score"].mean(), 1),
    "distribuicao_status": df_resultados["status"].value_counts().to_dict(),
    "pior_uc": df_resultados.loc[
        df_resultados["score"].idxmin(),
        ["cod_consumidor", "score", "problemas"]
    ].to_dict(),
}

prompt_relatorio = f"""
Você é um analista de dados sênior de uma distribuidora de energia elétrica.

Com base na análise de qualidade abaixo, escreva um relatório executivo em português.
O relatório deve ter: situação geral, principais riscos e recomendações priorizadas.
Seja direto e use linguagem adequada para gestores (sem jargões técnicos de TI).
Máximo 3 parágrafos.

Dados da análise:
{json.dumps(resumo_base, ensure_ascii=False, indent=2, default=str)}
"""

response = client.models.generate_content(
    model=MODEL,
    contents=prompt_relatorio,
    config=types.GenerateContentConfig(temperature=0.3),
)

print("RELATÓRIO EXECUTIVO — QUALIDADE DE DADOS DE CONSUMIDORES")
print("=" * 60)
print(response.text)
print("=" * 60)
print(f"\nTokens usados: entrada={response.usage_metadata.prompt_token_count} | saída={response.usage_metadata.candidates_token_count}")

## 4. Conversa multi-turno com histórico

O Gemini suporta sessões de chat que mantêm o histórico automaticamente entre as mensagens.

In [ ]:
# Criar uma sessão de chat — o histórico é mantido automaticamente
chat = client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        system_instruction="Você é um analista de dados do setor elétrico brasileiro. "
                           "Seja objetivo e foque em impactos para o negócio.",
        temperature=0.2,
    ),
)

# Turno 1: apresentar os dados
contexto = f"""Tenho os seguintes dados de qualidade de consumidores analisados:\n\n{df_resultados.to_markdown(index=False)}\n\nQual é o panorama geral da qualidade?"""

print("TURNO 1")
r1 = chat.send_message(contexto)
print(f"Gemini: {r1.text}\n")

# Turno 2: pergunta de acompanhamento (sem repetir os dados)
print("TURNO 2")
r2 = chat.send_message("Quais devo priorizar para correção e por quê?")
print(f"Gemini: {r2.text}\n")

# Turno 3: pedir SQL
print("TURNO 3")
r3 = chat.send_message(
    "Gere uma query SQL Snowflake para identificar UCs com score abaixo de 75 "
    "na tabela CONSUMIDORES_UC. Use CTE e comente em português."
)
print(f"Gemini: {r3.text}")

## 5. Inspecionando o histórico da conversa

In [ ]:
# O objeto chat mantém o histórico completo acessível
print(f"Total de turnos na conversa: {len(chat.get_history())}\n")

for i, msg in enumerate(chat.get_history()):
    papel = "Usuário" if msg.role == "user" else "Gemini"
    texto = msg.parts[0].text[:120].replace("\n", " ")
    print(f"[{i+1}] {papel}: {texto}..." if len(msg.parts[0].text) > 120 else f"[{i+1}] {papel}: {texto}")

## 6. Dicas de uso da API em produção

| Situação | Abordagem recomendada |
|----------|----------------------|
| Retorno JSON obrigatório | `response_mime_type="application/json"` no config |
| Análise de registro único | `client.models.generate_content(...)` |
| Lote de registros | Loop com `time.sleep(0.5)` + try/except |
| Relatório narrativo | Consolidar dados antes de enviar ao modelo |
| Conversa com histórico | `client.chats.create()` + `chat.send_message()` |
| Custo controlado | Usar amostras em dev, cachear resultados no Snowflake |

> **Tokens:** monitore `response.usage_metadata.prompt_token_count` para controlar custos em lote.